# Muhr — Paper 1 baseline tajribalari (E2, E4c, U-cross)

Bu ikkinchi bloknot. U chuqur suv belgisi baseline'larini **bizning tasvirlar, hujumlar va sinxronizatsiya qidiruvimiz** bilan sinaydi. Baseline'lar Video Seal'ning rasmiy interfeysi orqali yuklanadi (facebookresearch/videoseal, commit `f12875a9`): HiDDeN, MBRS, CIN, TrustMark-Q, WAM (TorchScript, HF `tangtianzhong/img-wm-torchscript`), Video Seal 1.0 va Pixel Seal (rasmiy model kartalari).

**Input (Add Input):** `muhr-v2` (v2.2), `best.pt` manbasi, COCO 2017 (`awsaf49/coco-2017-dataset`).
**Sozlamalar:** GPU T4, **Internet: On** (GitHub, Hugging Face va dl.fbaipublicfiles'dan yuklab olish uchun).

| STAGE | Tajriba | Taxminiy vaqt |
|---|---|---|
| `bl_smoke` | har bir usul: 4 tasvir, toza / JPEG50 bit aniqligi, PSNR, tezlik | 5–10 daqiqa, **birinchi** |
| `bl_unmarked` | E4c: 5 000 belgisiz COCO test2017 + 300 degenerat; (i) oddiy dekodlash, (ii) bizning N_geo gipotezalarimiz | smoke'dan keyin hisoblanadi |
| `bl_marked` | E2 + U-cross: T-COCO'dan 1 000 tasvir, 27 hujum, sifat; bizning dekoder begona belgilarda | bajarildi |
| `bl_codec` | bizning kalitli kodek (G9) Pixel Seal va Video Seal'ning birinchi 128 bitida; 1 000 tasvir, 27 hujum | ≈ 1 GPU-soat [taxmin] |

`bl_smoke` dan keyin har bir usul uchun `OK` yoki `CHECK WEIGHTS!` qatorlarini menga yuboring. `acc_clean` 0.9 dan past bo'lsa, og'irliklar to'g'ri yuklanmagan bo'ladi: unda o'sha usul tuzatilmaguncha keyingi bosqichlarni ishga tushirmang.
Agar 11 soatga sig'masa: shu versiya chiqishini Add Input qilib, xuddi shu `STAGE` bilan qayta Commit qiling. Tayyor bo'laklar qayta hisoblanmaydi. `METHODS` orqali usullarni ikki sessiyaga bo'lish mumkin.


In [ ]:
# ===== Sozlamalar =====
STAGE = "bl_codec"      # bl_codec  (bajarilgan: bl_smoke, bl_unmarked, bl_marked)
METHODS = "hidden,mbrs,cin,trustmark,wam,videoseal,pixelseal"
MAX_HOURS = 11.0
VS_COMMIT = "f12875a9b5e711e0c6e2dc3a22aca18231ce4a1a"           # Video Seal kodi (2026-11-13 holati)
HF_REPO, HF_REV = "tangtianzhong/img-wm-torchscript", "845dc751783db2a03a4b14ea600b0a4a9aba89aa"


In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path
W, IN, TMP = Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/tmp")
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"buyruq xato bilan tugadi: {cmd[:3]}")

def find_dir(name):
    hits = [h for h in sorted(glob.glob(str(IN / "**" / name), recursive=True)) if os.path.isdir(h)]
    return Path(hits[0]) if hits else None

# --- bizning kod va model
def _ver(d):
    try:
        return tuple(int(x) for x in (d / "VERSION").read_text().strip().split("."))
    except Exception:
        return (0,)
src = sorted({Path(p).parents[1] for p in glob.glob(str(IN / "**/muhr_proj/muhr_exp/codec2.py"), recursive=True)},
             key=_ver, reverse=True)       # Input'da bir nechta nusxa bo'lsa, eng yangi versiya olinadi
assert src, "muhr-v2 dataseti (muhr_proj/muhr_exp) topilmadi — Add Input qiling"
print("kod:", src[0], "versiya", ".".join(map(str, _ver(src[0]))))
assert _ver(src[0]) >= (2, 4), "muhr-v2 eski versiyada — Input'da eng oxirgi versiyani (2.4) tanlang"
CODE = W / "muhr_proj"
if CODE.exists():
    shutil.rmtree(CODE)                     # avvalgi ishdan qolgan eski nusxani ishlatmaslik uchun
shutil.copytree(src[0], CODE)
ck = sorted(glob.glob(str(IN / "**/runs/taklif/best.pt"), recursive=True))
assert ck, "runs/taklif/best.pt topilmadi"
CKPT = ck[0]; print("checkpoint:", CKPT)
run([sys.executable, "-m", "pip", "install", "-q", "galois", "lpips", "bchlib", "omegaconf", "einops", "pytorch_msssim",
     "pycocotools", "av", "decord"])


In [ ]:
# --- Video Seal kodi va baseline og'irliklari
VS = TMP / "videoseal"
if not VS.exists():
    run(["git", "clone", "-q", "https://github.com/facebookresearch/videoseal", VS])
    run(["git", "checkout", "-q", VS_COMMIT], cwd=VS)
run(["git", "rev-parse", "HEAD"], cwd=VS)
(VS / "ckpts").mkdir(exist_ok=True)
if not (VS / "ckpts" / "hidden_encoder_48b.pt").exists():
    from huggingface_hub import snapshot_download
    try:
        d = snapshot_download(HF_REPO, revision=HF_REV, local_dir=str(TMP / "hf_ts"))
    except Exception as e:
        print("pinned revision failed:", e, "-> main"); d = snapshot_download(HF_REPO, local_dir=str(TMP / "hf_ts"))
    for f in glob.glob(str(Path(d) / "**" / "*.pt"), recursive=True):
        shutil.copy(f, VS / "ckpts" / Path(f).name)
print(sorted(os.listdir(VS / "ckpts")))
# Video Seal / Pixel Seal kartalari og'irlikni birinchi yuklashda o'zi ckpts/ ga tushiradi


In [ ]:
# --- ma'lumotlar (asosiy bloknot bilan bir xil bo'linish)
def link_subset(files, dst):
    dst = Path(dst)
    if dst.exists():
        return dst
    dst.mkdir(parents=True)
    for f in files:
        os.symlink(f, dst / Path(f).name)
    return dst
val = find_dir("val2017"); assert val is not None, "COCO val2017 topilmadi"
vfiles = sorted(glob.glob(str(val / "*.jpg")))
T_COCO = link_subset(vfiles[1500:3500], TMP / "test_coco")        # e3_coco bilan bir xil tasvirlar
TEST2017 = find_dir("test2017"); assert TEST2017 is not None, "COCO test2017 topilmadi"

OUT = W / STAGE
OUT.mkdir(exist_ok=True)
prev = glob.glob(str(IN / "**" / f"{STAGE}.zip"), recursive=True)
if prev and not any(OUT.iterdir()):
    print("oldingi natija topildi:", prev[0]); shutil.unpack_archive(prev[0], OUT)
print("mavjud bo'laklar:", len(glob.glob(str(OUT / "**" / "chunk_*.npz"), recursive=True)))


In [ ]:
os.chdir(CODE)
py, common = sys.executable, ["--vs-root", VS, "--ckpt", CKPT, "--methods", METHODS, "--out", OUT,
                              "--max-hours", MAX_HOURS]
if STAGE == "bl_smoke":
    run([py, "scripts/e2_baselines.py", "--stage", "smoke", "--data", T_COCO, "--n", 4, *common])
elif STAGE == "bl_unmarked":      # E4c: e4_unmarked bilan bir xil test2017 tasvirlari (birinchi 5 000)
    run([py, "scripts/e2_baselines.py", "--stage", "unmarked", "--data", TEST2017, "--n", 5000,
         "--degenerate", 300, "--chunk", 250, *common])
elif STAGE == "bl_marked":        # E2 + U-cross: e3_coco tasvirlarining birinchi 1 000 tasi
    run([py, "scripts/e2_baselines.py", "--stage", "marked", "--data", T_COCO, "--n", 1000, "--chunk", 50, *common])
elif STAGE == "bl_codec":         # bizning kalitli kodek (G9) Pixel Seal / Video Seal ichida, xuddi shu 1 000 tasvir
    common[common.index("--methods") + 1] = "pixelseal,videoseal"
    run([py, "scripts/e2_baselines.py", "--stage", "marked", "--codec", "G9", "--data", T_COCO, "--n", 1000,
         "--chunk", 50, *common])
else:
    raise SystemExit("noma'lum STAGE")


In [ ]:
os.chdir(W)
z = shutil.make_archive(str(W / STAGE), "zip", OUT)
print("saqlandi:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
